## Import necessary modules

In [3]:
from transformers import AutoTokenizer,AutoModelForSeq2SeqLM,pipeline,AutoModelForCausalLM
import torch
import ast

c:\Users\harsh\Harshit\ML-CaPsule\Automatic Comment Generation\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Read The sample code file

In [4]:

with open("../data/sample_code_snippets.py") as file:
    code = file.read()
    tree = ast.parse(code)
    functions =[]
    for node in tree.body:
        if isinstance(node,ast.FunctionDef):
            fn_code = ast.get_source_segment(code,node)
            functions.append(fn_code)
    for func in functions:
        print(func)
        print("-"*50)

def add_numbers(a, b):
    return a + b
--------------------------------------------------
def factorial(n):
    if n == 0:
        return 1
    return n * factorial(n - 1)
--------------------------------------------------
def is_prime(num):

    if num <= 1:
        return False

    for i in range(2, int(num ** 0.5) + 1):

        if num % i == 0:
            return False

    return True
--------------------------------------------------
def reverse_string(text):
    return text[::-1]
--------------------------------------------------
def count_vowels(sentence):

    vowels = "aeiouAEIOU"

    return sum(
        1 for char in sentence
        if char in vowels
    )
--------------------------------------------------
def find_max(numbers):

    maximum = numbers[0]

    for num in numbers:

        if num > maximum:
            maximum = num

    return maximum
--------------------------------------------------
def fibonacci(n):

    sequence = [0, 1]

    while len(sequence) < n:


## Initialize the Pretrained Model

In [5]:
model_name = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float32
)

Loading weights: 100%|██████████| 338/338 [00:07<00:00, 46.42it/s]


## Generate Doc Strings with the Model

In [8]:
with open("../outputs/generated_docstrings.txt","w",encoding="utf-8") as file:
    for function_code in functions:

        prompt = f'''
        Python:{function_code}

        """
        '''
        inputs = tokenizer.encode(
            prompt,
            return_tensors = "pt",
            truncation = True,
            max_length = 512
        )

        outputs = model.generate(
            inputs,
            max_new_tokens=40,
            temperature=0.1,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
            
        )

        generated= tokenizer.decode(
            outputs[0],
            skip_special_tokens = True
        )
        generated = generated[len(prompt):]

        generated = generated.split('"""')[0]

        generated = generated.split("*/")[0]

        generated = generated.split("\ndef")[0]

        generated = generated.split("\n")[0]

        generated = generated.strip()

        file.write("FUNCTION:\n")
        file.write(function_code)
        file.write("\n\n")

        file.write("GENERATED DOCSTRING:\n")
        file.write(generated)
        file.write("\n")

        file.write("=" * 80)
        file.write("\n\n")